# claims-support-rl on Colab: 14B GRPO with Claude as judge

Runtime: **A100 80GB** (Runtime > Change runtime type). Secrets: add `ANTHROPIC_API_KEY` under the key icon on the left before running.

Runs 8 and 9 train Qwen2.5-14B-Instruct (bf16, LoRA) with prompt v5 and v8 on the relabeled sub35, judged by `claude-haiku-4-5`. Their rubric scores are not comparable with runs 3-7 (judged by qwen3:30b-a3b on the home box); route accuracy is.

In [19]:
%pip install -q "torch>=2.4" "transformers==5.17.0" "trl==1.13.0" "peft==0.20.0" accelerate datasets "anthropic==1.4.0"
# Colab ships torchao 0.10, which peft 0.20 rejects on import (needs >=0.16). peft uses it only for
# quantized layers, which this LoRA setup does not touch; peft imports cleanly without it.
%pip uninstall -y -q torchao
import torch, transformers, trl, peft, anthropic
print(torch.__version__, transformers.__version__, trl.__version__, peft.__version__, anthropic.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 117.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.5 MB/s eta 0:00:00
2.11.0+cu128 5.17.0 1.13.0 0.20.0 1.4.0
NVIDIA A100-SXM4-80GB, 81920 MiB


## Code and data

Everything the notebook needs is in the public repo at commit `a4d860268b35` (reward/, train/, eval/, the KB versions, the relabeled sub35, the stratified test). The next cell clones it and checks that commit out.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/claims-support-rl'
import os; os.makedirs(DRIVE, exist_ok=True)

%cd /content
!test -d claims-support-rl || git clone https://github.com/aeril716/claims-support-rl.git
%cd /content/claims-support-rl
!git fetch -q origin && git checkout -q 404dd6c && git log --oneline -1
!ls eval/pass_at_k.py

Mounted at /content/drive
/content
Cloning into 'claims-support-rl'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 158 (delta 68), reused 142 (delta 52), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 181.07 KiB | 18.11 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/content/claims-support-rl
404dd6c (HEAD) Add pass@k route eval and the 14B base results on the stratified test (v5, v8)
eval/pass_at_k.py


## Judge: Claude through the Messages API

In [2]:
from google.colab import userdata
import os
os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
os.environ['JUDGE_BACKEND'] = 'anthropic'

import sys; sys.path.insert(0, '/content/claims-support-rl')
from reward import judge
print(judge.judge_info())
verdict, reasoning = judge.ask_with_reasoning(
    'Does the reply state the 31-day rule?',
    'Customer message: "I spilled coffee on my laptop 20 days after signing up."\n\n'
    'Support agent reply: "Your plan is active, but repair and replacement coverage starts on day 31 after enrollment."')
print('verdict:', verdict, '| reasoning:', reasoning, '| usage:', judge.LAST)

ModuleNotFoundError: No module named 'reward'

## Baseline: 14B base, prompt v5, stratified test (judge = anthropic)

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v5 --score \
    --out "$DRIVE/eval_base14b_v5_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

## run8: 14B, prompt v5, relabeled sub35

run4 settings: 8 generations, batch 2 x accum 4, beta 0, seed 42, 35 steps, default reward weights, LoRA r=16 on the seven projection modules. Checkpoints, completions and live_steps go to Drive. The log prints s/step and peak VRAM after step 2.

In [ ]:
%env JUDGE_BACKEND=anthropic
!python train/train_grpo.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v5 \
    --out "$DRIVE/grpo_run8_14b_v5" --batch 2 --accum 4 --gens 8 --beta 0 --steps 35 --save-steps 35 --seed 42 \
    --tasks data/v3_kb_definitions/tasks_train_sub35.jsonl 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

## run8 eval: checkpoint-35, v5, stratified test

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v5 --score \
    --adapter "$DRIVE/grpo_run8_14b_v5/checkpoint-35" --out "$DRIVE/eval_run8_ckpt35_v5_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

In [ ]:
import json
def table(pairs):
    sums = {n: json.load(open(f'{DRIVE}/{d}/summary.json')) for n, d in pairs}
    routes = ['ask_question','file_claim','explain_not_covered','escalate','refer_to_manufacturer','explain_waiting_period','tech_support']
    print('| gold route | ' + ' | '.join(n for n, _ in pairs) + ' |'); print('|---' * (len(pairs) + 1) + '|')
    for r in routes:
        print(f'| {r} | ' + ' | '.join(f"{sums[n]['per_route'][r]['correct']}/{sums[n]['per_route'][r]['total']}" for n, _ in pairs) + ' |')
    print('| **total** | ' + ' | '.join(f"**{round(sums[n]['route_accuracy']*40)}/40**" for n, _ in pairs) + ' |')
    print('| rubric pass rate | ' + ' | '.join(f"{sums[n].get('rubric_pass_rate_mean', float('nan')):.3f}" for n, _ in pairs) + ' |')
    print('judge:', {n: sums[n].get('judge') for n, _ in pairs})
table([('14B base v5', 'eval_base14b_v5_test3'), ('run8 ckpt35 v5', 'eval_run8_ckpt35_v5_test3')])

## run9: same with prompt v8

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --score \
    --out "$DRIVE/eval_base14b_v8_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

In [ ]:
%env JUDGE_BACKEND=anthropic
!python train/train_grpo.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 \
    --out "$DRIVE/grpo_run9_14b_v8" --batch 2 --accum 4 --gens 8 --beta 0 --steps 35 --save-steps 35 --seed 42 \
    --tasks data/v3_kb_definitions/tasks_train_sub35.jsonl 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

In [ ]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --score \
    --adapter "$DRIVE/grpo_run9_14b_v8/checkpoint-35" --out "$DRIVE/eval_run9_ckpt35_v8_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'
table([('14B base v8', 'eval_base14b_v8_test3'), ('run9 ckpt35 v8', 'eval_run9_ckpt35_v8_test3')])

## 14B test V8 PASS@K sub35

In [4]:
!python eval/pass_at_k.py --help

usage: pass_at_k.py [-h] --model MODEL
                    [--prompt-version {v1,v2,v3,v4,v5,v6,v7,v7run7,v8}]
                    [--tasks TASKS] [--k K] [--temperature TEMPERATURE]
                    --out OUT [--greedy GREEDY] [--bf16] [--seed SEED]

options:
  -h, --help            show this help message and exit
  --model MODEL
  --prompt-version {v1,v2,v3,v4,v5,v6,v7,v7run7,v8}
  --tasks TASKS         task file (default: the test split)
  --k K
  --temperature TEMPERATURE
                        sampling temperature; the GRPO trainer uses 1.0
                        (train/train_grpo.py)
  --out OUT             jsonl, one line per task
  --greedy GREEDY       summary.json of the matching eval/before_after.py run,
                        for the reference column
  --bf16                bfloat16 instead of float16 (A100/H100)
  --seed SEED


In [4]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 \
    --tasks data/v3_kb_definitions/tasks_train_sub35.jsonl --k 16 --temperature 1.0 \
    --out "$DRIVE/passk_base14b_v8_sub35.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa | prompt v8 | k=16 temperature=1.0 top_p=1.0 max_new_tokens=512 | 35 tasks
  1/35 t206 ask_question             gold in 12/16
  2/35 t047 explain_waiting_period   gold in 9/16
  3/35 t231 ask_question             gold in 0/16   MISS
  4/35 t208 ask_question             gold in 1/16
  5/35 t338 ask_question             gold in 15/16
  6/35 t065 explain_waiting_period   gold in 16/16
  7/35 t193 tech_support             gold in 16/16
  8/35 t185 tech_support             gold in 16/16
  9/35 t106 refer_to_manufacturer    gold in 16/16
  10/35 t048 escalate                 gold in 0/16   MISS
  11/35 t162 explain_not_covered      gold in 16/16
  12/35 t241 ask_question             gold in 0/16   MISS
  13/35 t090 file_claim               gold in 14/16
  14/35 t411 tech_support             gold in 16/16
  15/35 t014 escalate                 gold in 0/16   MISS
  16/35 t103 refer_to_manufacturer    gold in 16/16
  17/35 t101 refer_to_manufac

In [5]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/claims-support-rl'
import os; os.makedirs(DRIVE, exist_ok=True)

%cd /content
!test -d claims-support-rl || git clone https://github.com/aeril716/claims-support-rl.git
%cd /content/claims-support-rl
!git fetch -q origin && git checkout -q d62b3cb && git log --oneline -1
!python eval/pass_at_k.py --help | grep adapter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/claims-support-rl
d62b3cb (HEAD, origin/main, origin/HEAD, main) pass@k: --adapter for checkpoints; passk_buckets.py groups tasks by gold hits
                    --out OUT [--greedy GREEDY] [--adapter ADAPTER] [--bf16]
  --adapter ADAPTER     LoRA checkpoint directory to load on the base, as
                        eval/before_after.py --adapter


In [7]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [8]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 \
    --tasks data/v3_kb_definitions/tasks_train_sub35.jsonl --k 16 --temperature 1.0 \
    --adapter "$DRIVE/grpo_run9_14b_v8/checkpoint-35" \
    --out "$DRIVE/passk_run9_v8_sub35.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa + adapter /content/drive/MyDrive/claims-support-rl/grpo_run9_14b_v8/checkpoint-35 | prompt v8 | k=16 temperature=1.0 top_p=1.0 max_new_tokens=512 | 35 tasks
  1/35 t206 ask_question             gold in 12/16
  2/35 t047 explain_waiting_period   gold in 12/16
  3/35 t231 ask_question             gold in 0/16   MISS
  4/35 t208 ask_question             gold in 1/16
  5/35 t338 ask_question             gold in 15/16
  6/35 t065 explain_waiting_period   gold in 16/16
  7/35 t193 tech_support             gold in 16/16
  8/35 t185 tech_support             gold in 16/16
  9/35 t106 refer_to_manufacturer    gold in 16/16
  10/35 t048 escalate                 gold in 0/16   MISS
  11/35 t162 explain_not_covered      gold in 16/16
  12/35 t241 ask_question             gold in 0/16   MISS
  13/35 t090 file_claim               gold in 16/16
  14/35 t411 tech_support             gold in 16/16
  15/35 t014 escalate                 gold in 0/16   MIS

In [9]:
!python eval/passk_buckets.py "$DRIVE/passk_base14b_v8_sub35.jsonl" "$DRIVE/passk_run9_v8_sub35.jsonl"

/content/drive/MyDrive/claims-support-rl/passk_base14b_v8_sub35.jsonl: 35 tasks, k=16, prompt v8
| gold hits of 16 | tasks | by gold route |
|---|---|---|
| 0 | 12 | escalate 5, ask_question 4, refer_to_manufacturer 1, explain_not_covered 1, file_claim 1 |
| 1-3 | 3 | ask_question 2, explain_not_covered 1 |
| 4-12 | 5 | ask_question 2, explain_waiting_period 2, file_claim 1 |
| 13-15 | 3 | ask_question 1, file_claim 1, explain_not_covered 1 |
| 16/16 | 12 | tech_support 5, refer_to_manufacturer 4, explain_waiting_period 1, explain_not_covered 1, file_claim 1 |

4-12 (mixed groups):
  t206  ask_question             12/16
  t290  ask_question             9/16
  t047  explain_waiting_period   9/16
  t087  explain_waiting_period   5/16
  t060  file_claim               5/16

/content/drive/MyDrive/claims-support-rl/passk_run9_v8_sub35.jsonl: 35 tasks, k=16, adapter /content/drive/MyDrive/claims-support-rl/grpo_run9_14b_v8/checkpoint-35, prompt v8
| gold hits of 16 | tasks | by gold route |


# 14b v8 + reasoning for DPO

student samples for DPO: 14B base, prompt v8 + reasoning field, train 320

Same base and dtype as every training run (bf16). k=4 samples per task at
temperature 1.0, raw completion text saved. Purpose: pool of student-written
completions for DPO pairs — route-wrong samples become rejected, route-right
samples become chosen (mixed with Sonnet 5 chosen from
data/sft/train_v8reason.jsonl). Output goes to Drive as
student_v8reason_train_k4.jsonl. No judge calls.

In [13]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/claims-support-rl'
import os; os.makedirs(DRIVE, exist_ok=True)

%cd /content
!test -d claims-support-rl || git clone https://github.com/aeril716/claims-support-rl.git
%cd /content/claims-support-rl
!git fetch -q origin && git checkout -q c5d734c && git log --oneline -3

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/claims-support-rl
c5d734c (HEAD) Prompt renderer: allow --reasoning on v5-v8 by inserting the reasoning line before route
b2dc4aa pass@k: --reasoning renders the reasoning-field prompt; --save-raw stores every completion text
d62b3cb (main) pass@k: --adapter for checkpoints; passk_buckets.py groups tasks by gold hits


In [14]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --tasks data/v3_kb_definitions/tasks_train.jsonl --k 4 --temperature 1.0 --save-raw --max-new-tokens 768 \
    --out "$DRIVE/student_v8reason_train_k4.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa | prompt v8 + reasoning | k=4 temperature=1.0 top_p=1.0 max_new_tokens=768 | 307 tasks
  1/307 t199 ask_question             gold in 0/4   MISS
  2/307 t200 ask_question             gold in 4/4
  3/307 t201 ask_question             gold in 0/4   MISS
  4/307 t202 ask_question             gold in 1/4
  5/307 t203 ask_question             gold in 3/4
  6/307 t205 ask_question             gold in 1/4
  7/307 t206 ask_question             gold in 2/4
  8/307 t207 ask_question             gold in 0/4   MISS
  9/307 t208 ask_question             gold in 0/4   MISS
  10/307 t209 ask_question             gold in 3/4
  11/307 t212 ask_question             gold in 0/4   MISS
  12/307 t213 ask_question             gold in 0/4   MISS
  13/307 t214 ask_question             gold in 4/4
  14/307 t215 ask_question             gold in 3/4
  15/307 t216 ask_question             gold in 0/4   MISS
  16/307 t218 ask_question             gold in 0/4   MISS

In [15]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --k 16 --temperature 1.0 --max-new-tokens 768 \
    --out "$DRIVE/passk_base14b_v8reason_test3.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa | prompt v8 + reasoning | k=16 temperature=1.0 top_p=1.0 max_new_tokens=768 | 40 tasks
  1/40 t387 ask_question             gold in 16/16
  2/40 t388 ask_question             gold in 16/16
  3/40 t389 ask_question             gold in 16/16
  4/40 t391 ask_question             gold in 16/16
  5/40 t392 ask_question             gold in 16/16
  6/40 t398 ask_question             gold in 16/16
  7/40 t120 file_claim               gold in 15/16
  8/40 t128 file_claim               gold in 15/16
  9/40 t130 file_claim               gold in 15/16
  10/40 t147 file_claim               gold in 13/16
  11/40 t171 file_claim               gold in 15/16
  12/40 t174 file_claim               gold in 12/16
  13/40 t173 explain_not_covered      gold in 15/16
  14/40 t423 explain_not_covered      gold in 13/16
  15/40 t441 explain_not_covered      gold in 15/16
  16/40 t442 explain_not_covered      gold in 8/16
  17/40 t443 explain_not_covered      go

# run10: RL only, prompt v8 + reasoning field, mixed-103 training set

Same as run9 (14B bf16, LoRA r=16, batch 2 x accum 4, 8 generations, beta 0,
seed 42, Haiku judge) with two changes: the reasoning field is on, and the
training set is the 103 train tasks where the base had at least one right
and one wrong sample at k=4 (tasks_train_mixed.jsonl, built in the cell
below from student_v8reason_train_k4.jsonl). 52 steps = one pass over 103
tasks at batch 2. Checkpoint and completions go to Drive.

In [16]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/claims-support-rl'
import os; os.makedirs(DRIVE, exist_ok=True)

%cd /content
!test -d claims-support-rl || git clone https://github.com/aeril716/claims-support-rl.git
%cd /content/claims-support-rl
!git fetch -q origin && git checkout -q fc1c233 && git log --oneline -1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/claims-support-rl
fc1c233 (HEAD, origin/main, origin/HEAD) Teacher dataset r3: claim-limit and not-covered instructions on the teacher side; --task-ids and repeatable --teacher-suffix


In [17]:
import json
rows = [json.loads(l) for l in open(f"{DRIVE}/student_v8reason_train_k4.jsonl")]
mixed = {r["task_id"] for r in rows
         if 0 < sum(1 for s in r["samples"] if s == r["gold"]) < len(r["samples"])}
tasks = [json.loads(l) for l in open("data/v3_kb_definitions/tasks_train.jsonl")]
sub = [t for t in tasks if t["task_id"] in mixed]
out = "data/v3_kb_definitions/tasks_train_mixed.jsonl"
with open(out, "w") as f:
    for t in sub: f.write(json.dumps(t) + "\n")
!cp {out} "$DRIVE/tasks_train_mixed.jsonl"
from collections import Counter
print(len(sub), dict(sorted(Counter(t["route_answer"] for t in sub).items())))

103 {'ask_question': 41, 'escalate': 6, 'explain_not_covered': 11, 'explain_waiting_period': 8, 'file_claim': 14, 'refer_to_manufacturer': 13, 'tech_support': 10}


In [20]:
%env JUDGE_BACKEND=anthropic
!python train/train_grpo.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --out "$DRIVE/grpo_run10_14b_v8reason" --batch 2 --accum 4 --gens 8 --beta 0 --steps 52 --save-steps 52 --seed 42 \
    --tasks data/v3_kb_definitions/tasks_train_mixed.jsonl 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Streaming output truncated to the last 5000 lines.
│ │ dents; device │               │              │               │           │ │
│ │ works         │               │              │               │           │ │
│ │ normally      │               │              │               │           │ │
│ │ - damage      │               │              │               │           │ │
│ │ happened      │               │              │               │           │ │
│ │ before        │               │              │               │           │ │
│ │ enrollment or │               │              │               │           │ │
│ │ during        │               │              │               │           │ │
│ │ waiting       │               │              │               │           │ │
│ │ period        │               │              │               │           │ │
│ │ - app, OS,    │               │              │               │           │ │
│ │ virus, or     │               │              │        

In [21]:
%env JUDGE_BACKEND=anthropic
!git fetch -q origin && git checkout -q c3961e8 && git log --oneline -1
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning --score \
    --adapter "$DRIVE/grpo_run10_14b_v8reason/checkpoint-52" --out "$DRIVE/eval_run10_ckpt52_v8reason_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

env: JUDGE_BACKEND=anthropic
c3961e8 (HEAD, origin/main, origin/HEAD) SFT arm: LoRA on Qwen2.5-14B-Instruct from the r3 teacher dataset, QLoRA fallback on OOM
  1/40 t387
  2/40 t388
  3/40 t389
  4/40 t391
  5/40 t392
  6/40 t398
  7/40 t120
  8/40 t128
  9/40 t130
  10/40 t147
  11/40 t171
  12/40 t174
  13/40 t173
  14/40 t423
  15/40 t441
  16/40 t442
  17/40 t443
  18/40 t444
  19/40 t013
  20/40 t059
  21/40 t137
  22/40 t160
  23/40 t169
  24/40 t180
  25/40 t112
  26/40 t119
  27/40 t122
  28/40 t142
  29/40 t143
  30/40 t144
  31/40 t005
  32/40 t022
  33/40 t044
  34/40 t069
  35/40 t170
  36/40 t406
  37/40 t415
  38/40 t420
  39/40 t421
  40/40 t422

Qwen/Qwen2.5-14B-Instruct  (hf torch.bfloat16 cuda greedy + adapter /content/drive/MyDrive/claims-support-rl/grpo_run10_14b_v8reason/checkpoint-52)  11.8 min
  parsed raw 40/40   after repair 40/40
  route accuracy 35/40
    ask_question              6/6
    escalate                  4/6
    explain_not_covered       5/6
    ex

In [22]:
%env JUDGE_BACKEND=anthropic
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning --score \
    --out "$DRIVE/eval_base14b_v8reason_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

env: JUDGE_BACKEND=anthropic
  1/40 t387
  2/40 t388
  3/40 t389
  4/40 t391
  5/40 t392
  6/40 t398
  7/40 t120
  8/40 t128
  9/40 t130
  10/40 t147
  11/40 t171
  12/40 t174
  13/40 t173
  14/40 t423
  15/40 t441
  16/40 t442
  17/40 t443
  18/40 t444
  19/40 t013
  20/40 t059
  21/40 t137
  22/40 t160
  23/40 t169
  24/40 t180
  25/40 t112
  26/40 t119
  27/40 t122
  28/40 t142
  29/40 t143
  30/40 t144
  31/40 t005
  32/40 t022
  33/40 t044
  34/40 t069
  35/40 t170
  36/40 t406
  37/40 t415
  38/40 t420
  39/40 t421
  40/40 t422

Qwen/Qwen2.5-14B-Instruct  (hf torch.bfloat16 cuda greedy)  5.7 min
  parsed raw 40/40   after repair 40/40
  route accuracy 35/40
    ask_question              6/6
    escalate                  4/6
    explain_not_covered       5/6
    explain_waiting_period    5/5
    file_claim                6/6
    refer_to_manufacturer     4/6
    tech_support              5/5
  chosen routes: {'ask_question': 6, 'file_claim': 9, 'explain_not_covered': 5, 'escalate'

In [23]:
%env JUDGE_BACKEND=anthropic
!git fetch -q origin && git checkout -q c3961e8 && git log --oneline -1
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning --score \
    --adapter "$DRIVE/grpo_run10_14b_v8reason/checkpoint-52" --out "$DRIVE/eval_run10_ckpt52_v8reason_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

env: JUDGE_BACKEND=anthropic
c3961e8 (HEAD, origin/main, origin/HEAD) SFT arm: LoRA on Qwen2.5-14B-Instruct from the r3 teacher dataset, QLoRA fallback on OOM
  1/40 t387
  2/40 t388
  3/40 t389
  4/40 t391
  5/40 t392
  6/40 t398
  7/40 t120
  8/40 t128
  9/40 t130
  10/40 t147
  11/40 t171
  12/40 t174
  13/40 t173
  14/40 t423
  15/40 t441
  16/40 t442
  17/40 t443
  18/40 t444
  19/40 t013
  20/40 t059
  21/40 t137
  22/40 t160
  23/40 t169
  24/40 t180
  25/40 t112
  26/40 t119
  27/40 t122
  28/40 t142
  29/40 t143
  30/40 t144
  31/40 t005
  32/40 t022
  33/40 t044
  34/40 t069
  35/40 t170
  36/40 t406
  37/40 t415
  38/40 t420
  39/40 t421
  40/40 t422

Qwen/Qwen2.5-14B-Instruct  (hf torch.bfloat16 cuda greedy + adapter /content/drive/MyDrive/claims-support-rl/grpo_run10_14b_v8reason/checkpoint-52)  11.7 min
  parsed raw 40/40   after repair 40/40
  route accuracy 35/40
    ask_question              6/6
    escalate                  4/6
    explain_not_covered       5/6
    ex

In [24]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --tasks data/v3_kb_definitions/tasks_train_mixed.jsonl --k 4 --temperature 1.0 --save-raw --max-new-tokens 768 \
    --adapter "$DRIVE/grpo_run10_14b_v8reason/checkpoint-52" \
    --out "$DRIVE/passk_run10_v8reason_mixed_k4.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa + adapter /content/drive/MyDrive/claims-support-rl/grpo_run10_14b_v8reason/checkpoint-52 | prompt v8 + reasoning | k=4 temperature=1.0 top_p=1.0 max_new_tokens=768 | 103 tasks
  1/103 t202 ask_question             gold in 1/4
  2/103 t203 ask_question             gold in 2/4
  3/103 t205 ask_question             gold in 2/4
  4/103 t206 ask_question             gold in 4/4
  5/103 t209 ask_question             gold in 4/4
  6/103 t215 ask_question             gold in 3/4
  7/103 t221 ask_question             gold in 2/4
  8/103 t235 ask_question             gold in 0/4   MISS
  9/103 t247 ask_question             gold in 1/4
  10/103 t254 ask_question             gold in 2/4
  11/103 t255 ask_question             gold in 3/4
  12/103 t260 ask_question             gold in 3/4
  13/103 t265 ask_question             gold in 1/4
  14/103 t266 ask_question             gold in 1/4
  15/103 t272 ask_question             gold in 3/4
  16/103 t

In [25]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --k 16 --temperature 1.0 --max-new-tokens 768 \
    --adapter "$DRIVE/grpo_run10_14b_v8reason/checkpoint-52" \
    --out "$DRIVE/passk_run10_v8reason_test3.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa + adapter /content/drive/MyDrive/claims-support-rl/grpo_run10_14b_v8reason/checkpoint-52 | prompt v8 + reasoning | k=16 temperature=1.0 top_p=1.0 max_new_tokens=768 | 40 tasks
  1/40 t387 ask_question             gold in 16/16
  2/40 t388 ask_question             gold in 16/16
  3/40 t389 ask_question             gold in 16/16
  4/40 t391 ask_question             gold in 16/16
  5/40 t392 ask_question             gold in 16/16
  6/40 t398 ask_question             gold in 15/16
  7/40 t120 file_claim               gold in 15/16
  8/40 t128 file_claim               gold in 16/16
  9/40 t130 file_claim               gold in 15/16
  10/40 t147 file_claim               gold in 13/16
  11/40 t171 file_claim               gold in 16/16
  12/40 t174 file_claim               gold in 12/16
  13/40 t173 explain_not_covered      gold in 16/16
  14/40 t423 explain_not_covered      gold in 14/16
  15/40 t441 explain_not_covered      gold in 15/16
  

In [26]:
import json
ids = [json.loads(l)["task_id"] for l in open("data/v3_kb_definitions/tasks_train_mixed.jsonl")]
print(",".join(ids))

t202,t203,t205,t206,t209,t215,t221,t235,t247,t254,t255,t260,t265,t266,t272,t273,t285,t304,t306,t307,t312,t317,t318,t319,t320,t321,t322,t325,t326,t327,t330,t333,t335,t343,t347,t348,t351,t352,t353,t354,t356,t018,t024,t058,t079,t082,t091,t163,t164,t175,t176,t427,t431,t432,t433,t434,t435,t019,t023,t026,t070,t072,t085,t087,t089,t093,t096,t110,t117,t134,t135,t140,t145,t003,t009,t017,t021,t027,t031,t039,t042,t057,t060,t061,t062,t071,t084,t098,t104,t107,t124,t125,t136,t188,t191,t194,t198,t401,t405,t408,t410,t413,t414


# SFT arm eval: r3 teacher distillation, test 40, greedy + judge (rubric v1 and v2)

Adapter sft_r3_14b was trained on the server (fp16 LoRA r=16, 321 samples
from data/sft/train_v8reason_r3.jsonl capped at 50 per route, 3 epochs,
final loss 0.288, 3h19m) and copied to Drive. Evaluated here on bf16 like
every other arm. Baseline: 14B base with the reasoning field, 35/40,
rubric v2 0.847.

In [28]:
%env JUDGE_BACKEND=anthropic
!git fetch -q origin && git checkout -q a599b60 && git log --oneline -1
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning --score \
    --adapter "$DRIVE/sft_r3_14b" --out "$DRIVE/eval_sft_r3_v8reason_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

env: JUDGE_BACKEND=anthropic
a599b60 (HEAD, origin/main, origin/HEAD) sft.py: --bf16 flag for GPUs with bfloat16 (Colab A100)
  1/40 t387
  2/40 t388
  3/40 t389
  4/40 t391
  5/40 t392
  6/40 t398
  7/40 t120
  8/40 t128
  9/40 t130
  10/40 t147
  11/40 t171
  12/40 t174
  13/40 t173
  14/40 t423
  15/40 t441
  16/40 t442
  17/40 t443
  18/40 t444
  19/40 t013
  20/40 t059
  21/40 t137
  22/40 t160
  23/40 t169
  24/40 t180
  25/40 t112
  26/40 t119
  27/40 t122
  28/40 t142
  29/40 t143
  30/40 t144
  31/40 t005
  32/40 t022
  33/40 t044
  34/40 t069
  35/40 t170
  36/40 t406
  37/40 t415
  38/40 t420
  39/40 t421
  40/40 t422

Qwen/Qwen2.5-14B-Instruct  (hf torch.bfloat16 cuda greedy + adapter /content/drive/MyDrive/claims-support-rl/sft_r3_14b)  13.6 min
  parsed raw 40/40   after repair 40/40
  route accuracy 38/40
    ask_question              6/6
    escalate                  6/6
    explain_not_covered       6/6
    explain_waiting_period    5/5
    file_claim                5/

# SFT (sft_r3_14b) — per-task check on test 40

Reads outputs.json from the SFT eval above and prints, with the model's
reasoning, the three limit-plus-warranty cases (t137, t143, t144) plus any
file_claim / refer_to_manufacturer task the adapter got wrong.

Result: t137 escalate correct; t143 refer correct but the reasoning never
mentions the claim limit; t144 still escalate (sees both warranty and limit,
limit wins); t130 file_claim -> explain_not_covered (reasoning contradicts its
own conclusion). So the SFT did not learn "warranty active -> refer before
limit" — expected, since the r3 teacher had 0 kept samples for that case.
Teacher round r4 adds a rule-order line for the new103 batch.

In [31]:
import os
root = "/content/drive/MyDrive/claims-support-rl"
for d in sorted(os.listdir(root)): print(d)

eval_base14b_v5_test3
eval_base14b_v8_test3
eval_base14b_v8reason_test3
eval_run10_ckpt52_v8reason_test3
eval_run8_ckpt35_v5_test3
eval_run9_ckpt35_v8_test3
eval_sft_r3_v8reason_test3
grpo_run10_14b_v8reason
grpo_run11_14b_v8reason_sft
grpo_run8_14b_v5
grpo_run9_14b_v8
passk_base14b_v8_sub35.jsonl
passk_base14b_v8reason_test3.jsonl
passk_run10_v8reason_mixed_k4.jsonl
passk_run10_v8reason_test3.jsonl
passk_run9_v8_sub35.jsonl
sft_r3_14b
student_v8reason_train_k4.jsonl
tasks_train_mixed.jsonl


In [33]:
import json
path = "/content/drive/MyDrive/claims-support-rl/eval_sft_r3_v8reason_test3/outputs.json"
data = json.load(open(path))
rows = data if isinstance(data, list) else next(v for v in data.values() if isinstance(v, list))
print(rows[0].keys())
want = {"t137","t143","t144"}
for r in rows:
    tid = r.get("task_id") or r.get("id")
    gold = r.get("gold") or r.get("gold_route")
    chosen = r.get("chosen") or r.get("route") or r.get("pred")
    if tid in want or (gold in ("file_claim","refer_to_manufacturer") and chosen != gold):
        print(tid, gold, "->", chosen)
        print("   ", str(r.get("reasoning") or r.get("raw") or "")[:300])

dict_keys(['task_id', 'gold', 'chosen', 'correct', 'repairs', 'reasoning', 'facts', 'raw_output', 'reward', 'rubric_score_v2', 'items', 'rubric_score_v1', 'rubric_score'])
t130 file_claim -> explain_not_covered
    Tablet with a foggy screen caused by humidity/mold exposure falls under dust/heat/humidity failure, which is only covered after the manufacturer's warranty ends; warranty is inactive here.
t137 escalate -> escalate
    Tablet with a dust-related failure past the waiting period; manufacturer warranty inactive, so plan covers this, but claims_last_12m (2) equals the 2-claim limit for tablets, so no further claims can be accepted.
t143 refer_to_manufacturer -> refer_to_manufacturer
    Humidity failure on a phone is a dust/heat/humidity peril, which is only covered after the manufacturer warranty ends; here the warranty is still active, so it should be referred to the manufacturer.
t144 refer_to_manufacturer -> escalate
    Phone exposed to dust/heat causing failure; this is a 

# SFT (r4, new103) — LoRA on Colab A100, bf16

Trains the SFT-only arm and the init adapter for run11. Same recipe as the
server run that produced sft_r3_14b (train/sft.py, LoRA, 3 epochs, at most
2 samples per task and 50 per route), with two differences:
- data: data/sft/train_v8reason_r3_new103.jsonl (teacher r4 on the new103
  batch, disjoint from the RL set) instead of the r3 set on train 307
- --bf16 (A100) instead of fp16 (RTX 8000)

Output adapter: Drive sft_new103_14b. Next cells: SFT eval (greedy + rubric
v2 on test 40, --adapter sft_new103_14b), then run11 with --init-adapter.

In [37]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/claims-support-rl'
import os; os.makedirs(DRIVE, exist_ok=True)

%cd /content
!test -d claims-support-rl || git clone https://github.com/aeril716/claims-support-rl.git
%cd /content/claims-support-rl
!git fetch -q origin && git checkout -q 40c7718 && git log --oneline -1

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/claims-support-rl
40c7718 (HEAD, origin/main, origin/HEAD) sft.py: --data and --out flags, gold routes from the new103 task file too


In [38]:
%cd /content/claims-support-rl
!python train/sft.py --bf16 \
  --data data/sft/train_v8reason_r3_new103.jsonl \
  --out "$DRIVE/sft_new103_14b"

/content/claims-support-rl
data: /content/claims-support-rl/data/sft/train_v8reason_r3_new103.jsonl
used samples: /content/claims-support-rl/data/sft/used_r3_new103.jsonl
output dir: /content/drive/MyDrive/claims-support-rl/sft_new103_14b
samples whose route differs from gold (skipped): 0
| route | samples | tasks |
|---|---|---|
| ask_question | 17 | 9 |
| escalate | 29 | 15 |
| explain_not_covered | 20 | 10 |
| explain_waiting_period | 30 | 15 |
| file_claim | 26 | 13 |
| refer_to_manufacturer | 26 | 13 |
| tech_support | 26 | 14 |
| total | 174 | 89 |
rows after the per-task and per-route caps: 174
written: /content/claims-support-rl/data/sft/used_r3_new103.jsonl
=== starting bf16 run ===
[bf16] examples: 174, longest: 3059 tokens
Loading weights: 100% 579/579 [00:07<00:00, 73.46it/s]
trainable params: 68,812,800 || all params: 14,838,846,464 || trainable%: 0.4637
{'loss': '1.108', 'grad_norm': '0.9208', 'learning_rate': '0.0001', 'epoch': '0.04598'}
{'loss': '1.171', 'grad_norm': '

In [39]:
%env JUDGE_BACKEND=anthropic
!git fetch -q origin && git checkout -q 40c7718 && git log --oneline -1
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning --score \
    --adapter "$DRIVE/sft_new103_14b" --out "$DRIVE/eval_sft_new103_v8reason_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

env: JUDGE_BACKEND=anthropic
40c7718 (HEAD, origin/main, origin/HEAD) sft.py: --data and --out flags, gold routes from the new103 task file too
  1/40 t387
  2/40 t388
  3/40 t389
  4/40 t391
  5/40 t392
  6/40 t398
  7/40 t120
  8/40 t128
  9/40 t130
  10/40 t147
  11/40 t171
  12/40 t174
  13/40 t173
  14/40 t423
  15/40 t441
  16/40 t442
  17/40 t443
  18/40 t444
  19/40 t013
  20/40 t059
  21/40 t137
  22/40 t160
  23/40 t169
  24/40 t180
  25/40 t112
  26/40 t119
  27/40 t122
  28/40 t142
  29/40 t143
  30/40 t144
  31/40 t005
  32/40 t022
  33/40 t044
  34/40 t069
  35/40 t170
  36/40 t406
  37/40 t415
  38/40 t420
  39/40 t421
  40/40 t422

Qwen/Qwen2.5-14B-Instruct  (hf torch.bfloat16 cuda greedy + adapter /content/drive/MyDrive/claims-support-rl/sft_new103_14b)  13.2 min
  parsed raw 40/40   after repair 40/40
  route accuracy 40/40
    ask_question              6/6
    escalate                  6/6
    explain_not_covered       6/6
    explain_waiting_period    5/5
    file_c

# run11: SFT + RL — GRPO starting from the sft_new103_14b adapter

Same as run10 (v8 + reasoning, mixed-103 training set, 52 steps, batch 2 x
accum 4, 8 generations, beta 0, seed 42, Haiku judge, rubric v1 wording for
training) except the LoRA is initialised from the SFT adapter via
--init-adapter. The only factor that differs from run10 is the SFT init.

SFT adapter: sft_new103_14b (teacher r4 on tasks_sft_new103, disjoint from
the RL set; SFT-only eval 40/40, rubric v2 0.912). Do NOT use sft_r3_14b
here — it overlaps the RL set.

Output: grpo_run11_14b_sft_new103/checkpoint-52. Next: eval, then pass@16
for both sft_new103_14b and this checkpoint.

In [40]:
%env JUDGE_BACKEND=anthropic
!test -f data/v3_kb_definitions/tasks_train_mixed.jsonl || cp "$DRIVE/tasks_train_mixed.jsonl" data/v3_kb_definitions/
!python train/train_grpo.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --init-adapter "$DRIVE/sft_new103_14b" \
    --out "$DRIVE/grpo_run11_14b_sft_new103" --batch 2 --accum 4 --gens 8 --beta 0 --steps 52 --save-steps 52 --seed 42 \
    --tasks data/v3_kb_definitions/tasks_train_mixed.jsonl 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Streaming output truncated to the last 5000 lines.
│ │ dents; device │               │              │               │           │ │
│ │ works         │               │              │               │           │ │
│ │ normally      │               │              │               │           │ │
│ │ - damage      │               │              │               │           │ │
│ │ happened      │               │              │               │           │ │
│ │ before        │               │              │               │           │ │
│ │ enrollment or │               │              │               │           │ │
│ │ during        │               │              │               │           │ │
│ │ waiting       │               │              │               │           │ │
│ │ period        │               │              │               │           │ │
│ │ - app, OS,    │               │              │               │           │ │
│ │ virus, or     │               │              │        

In [41]:
%env JUDGE_BACKEND=anthropic
!git fetch -q origin && git checkout -q 40c7718 && git log --oneline -1
!python eval/before_after.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning --score \
    --adapter "$DRIVE/grpo_run11_14b_sft_new103/checkpoint-52" --out "$DRIVE/eval_run11_ckpt52_v8reason_test3" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

env: JUDGE_BACKEND=anthropic
40c7718 (HEAD) sft.py: --data and --out flags, gold routes from the new103 task file too
  1/40 t387
  2/40 t388
  3/40 t389
  4/40 t391
  5/40 t392
  6/40 t398
  7/40 t120
  8/40 t128
  9/40 t130
  10/40 t147
  11/40 t171
  12/40 t174
  13/40 t173
  14/40 t423
  15/40 t441
  16/40 t442
  17/40 t443
  18/40 t444
  19/40 t013
  20/40 t059
  21/40 t137
  22/40 t160
  23/40 t169
  24/40 t180
  25/40 t112
  26/40 t119
  27/40 t122
  28/40 t142
  29/40 t143
  30/40 t144
  31/40 t005
  32/40 t022
  33/40 t044
  34/40 t069
  35/40 t170
  36/40 t406
  37/40 t415
  38/40 t420
  39/40 t421
  40/40 t422

Qwen/Qwen2.5-14B-Instruct  (hf torch.bfloat16 cuda greedy + adapter /content/drive/MyDrive/claims-support-rl/grpo_run11_14b_sft_new103/checkpoint-52)  13.6 min
  parsed raw 40/40   after repair 40/40
  route accuracy 40/40
    ask_question              6/6
    escalate                  6/6
    explain_not_covered       6/6
    explain_waiting_period    5/5
    file_cl

In [42]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --k 16 --temperature 1.0 --max-new-tokens 768 \
    --adapter "$DRIVE/sft_new103_14b" \
    --out "$DRIVE/passk_sft_new103_v8reason_test3.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa + adapter /content/drive/MyDrive/claims-support-rl/sft_new103_14b | prompt v8 + reasoning | k=16 temperature=1.0 top_p=1.0 max_new_tokens=768 | 40 tasks
  1/40 t387 ask_question             gold in 14/16
  2/40 t388 ask_question             gold in 16/16
  3/40 t389 ask_question             gold in 16/16
  4/40 t391 ask_question             gold in 15/16
  5/40 t392 ask_question             gold in 16/16
  6/40 t398 ask_question             gold in 16/16
  7/40 t120 file_claim               gold in 16/16
  8/40 t128 file_claim               gold in 16/16
  9/40 t130 file_claim               gold in 14/16
  10/40 t147 file_claim               gold in 16/16
  11/40 t171 file_claim               gold in 15/16
  12/40 t174 file_claim               gold in 15/16
  13/40 t173 explain_not_covered      gold in 16/16
  14/40 t423 explain_not_covered      gold in 16/16
  15/40 t441 explain_not_covered      gold in 16/16
  16/40 t442 explain_not_

In [43]:
!python eval/pass_at_k.py --model Qwen/Qwen2.5-14B-Instruct --bf16 --prompt-version v8 --reasoning \
    --k 16 --temperature 1.0 --max-new-tokens 768 \
    --adapter "$DRIVE/grpo_run11_14b_sft_new103/checkpoint-52" \
    --out "$DRIVE/passk_run11_v8reason_test3.jsonl" 2>&1 | grep -vE 'Warning|Loading weights|Fetching'

Qwen/Qwen2.5-14B-Instruct torch.bfloat16 cuda sdpa + adapter /content/drive/MyDrive/claims-support-rl/grpo_run11_14b_sft_new103/checkpoint-52 | prompt v8 + reasoning | k=16 temperature=1.0 top_p=1.0 max_new_tokens=768 | 40 tasks
  1/40 t387 ask_question             gold in 16/16
  2/40 t388 ask_question             gold in 16/16
  3/40 t389 ask_question             gold in 16/16
  4/40 t391 ask_question             gold in 16/16
  5/40 t392 ask_question             gold in 16/16
  6/40 t398 ask_question             gold in 15/16
  7/40 t120 file_claim               gold in 16/16
  8/40 t128 file_claim               gold in 15/16
  9/40 t130 file_claim               gold in 15/16
  10/40 t147 file_claim               gold in 16/16
  11/40 t171 file_claim               gold in 16/16
  12/40 t174 file_claim               gold in 16/16
  13/40 t173 explain_not_covered      gold in 16/16
  14/40 t423 explain_not_covered      gold in 16/16
  15/40 t441 explain_not_covered      gold in 16/16
